# Week 2 - Task D: Feature Maps, Learned Filters, and Grad-CAM Interpretability

## 1. Interpretability in Deep Convolutional Networks
Deep neural networks are often mischaracterized as "black boxes." In this notebook, we visualize their internal representations and decision-making mechanisms through three complementary methods:
1. **Feature Map Activations**: Visualizing what intermediate convolutional channels detect when presented with an input image.
2. **Learned Convolutional Filters**: Examining the actual weights and spatial kernels learned by the network.
3. **Gradient-Weighted Class Activation Mapping (Grad-CAM)**: Using gradients of the target class score flowing into the final convolutional layer to highlight the exact spatial regions influencing the prediction.

### Mathematical Formulation of Grad-CAM
Let $y^c$ be the raw score (logit) for class $c$, and $A^k$ be the feature map activation of channel $k$ in the final convolutional layer.
1. **Neuron Importance Weights ($\alpha_k^c$)**: Computed via Global Average Pooling of gradients:
   $$\alpha_k^c = \frac{1}{Z} \sum_{i=1}^{U} \sum_{j=1}^{V} \frac{\partial y^c}{\partial A_{i,j}^k}$$
   where $Z = U \times V$ is the spatial area of the feature map.
2. **Class Activation Heatmap ($L_{\text{Grad-CAM}}^c$)**: Linear combination of feature maps weighted by $\alpha_k^c$, followed by a ReLU non-linearity:
   $$L_{\text{Grad-CAM}}^c = \text{ReLU}\left(\sum_{k} \alpha_k^c A^k\right)$$
   The ReLU ensures we only capture features that contribute positively to the class of interest.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import load_model

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")


In [ ]:
# Load the trained plant disease ResNet50 model
import sys
sys.path.append("..")
import model_utils

MODEL_PATH = os.path.join("models", "plant_disease_resnet50.keras")
model = model_utils.get_plant_disease_model(MODEL_PATH)
classes = model_utils.load_class_mapping()
print(f"Loaded model successfully. Number of classes: {len(classes)}")
for i, c in enumerate(classes):
    print(f"  [{i}] {c}")


## 2. Visualizing Intermediate Feature Maps
We inspect activations across three distinct stages of the network:
* **Early Conv Layer** (`conv1_conv`): Responds to low-level edges, oriented bars, and localized color contrasts.
* **Mid Conv Layer** (`conv3_block4_out`): Combines edges into motifs, leaf veins, and texture boundaries.
* **Late Conv Layer** (`conv5_block3_out`): Fires on class-specific semantic patterns (e.g. necrotic lesion spots, fungal concentric rings).


In [ ]:
# Select a sample test image (e.g. Early Blight)
sample_path = os.path.join("data", "plant_village_5_classes", "test", "Tomato___Early_blight")
sample_files = [f for f in os.listdir(sample_path) if f.lower().endswith(('.jpg', '.png'))]
img_path = os.path.join(sample_path, sample_files[0])

# Load and preprocess
raw_img = Image.open(img_path).convert('RGB').resize((224, 224))
input_tensor = preprocess_input(np.expand_dims(np.array(raw_img, dtype=np.float32), 0))

# Inspect layers
layer_names = ['conv1_conv', 'conv3_block4_out', 'conv5_block3_out']
intermediate_outputs = [model.get_layer(name).output for name in layer_names]
activation_model = tf.keras.models.Model(inputs=model.input, outputs=intermediate_outputs)

activations = activation_model.predict(input_tensor, verbose=0)

# Display feature maps for each layer
fig, axes = plt.subplots(3, 8, figsize=(16, 6))

for layer_idx, (layer_name, act) in enumerate(zip(layer_names, activations)):
    for channel_idx in range(8):
        ax = axes[layer_idx, channel_idx]
        ax.imshow(act[0, :, :, channel_idx], cmap='viridis')
        ax.axis('off')
        if channel_idx == 0:
            ax.set_title(f"{layer_name}", fontsize=10, fontweight='bold', loc='left')

plt.suptitle("Feature Maps: Early (Top) -> Mid (Middle) -> Late (Bottom)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 3. Visualizing Learned Convolutional Filters
We extract the learned weights from the first convolutional layer (`conv1_conv`).
Its shape is $(7, 7, 3, 64)$, corresponding to 64 distinct $7 \times 7$ kernels across the 3 RGB color channels.


In [ ]:
conv1_weights = model.get_layer('conv1_conv').get_weights()[0]
print(f"conv1_conv weight tensor shape: {conv1_weights.shape}")

# Plot 16 filter kernels
plt.figure(figsize=(10, 10))
for i in range(16):
    kernel = conv1_weights[:, :, :, i]
    # Normalize kernel to [0, 1] for visual display
    k_min, k_max = kernel.min(), kernel.max()
    kernel_norm = (kernel - k_min) / (k_max - k_min + 1e-8)
    
    plt.subplot(4, 4, i + 1)
    plt.imshow(kernel_norm)
    plt.title(f"Kernel #{i}", fontsize=9)
    plt.axis('off')

plt.suptitle("Learned conv1_conv Filters (7x7 RGB)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 4. Genuine Gradient-Based Grad-CAM Implementation
We implement the full mathematical Grad-CAM pipeline using `tf.GradientTape`.


In [ ]:
def make_gradcam_heatmap(img_tensor, model, last_conv_layer_name="conv5_block3_out", pred_index=None):
    # Construct a sub-model that outputs both the last conv activations and predictions
    last_conv_layer = model.get_layer(last_conv_layer_name)
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[last_conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_tensor)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # Compute gradients of top predicted class with respect to last conv feature maps
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # Global Average Pooling of gradients (alpha weights)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Linear combination: sum(alpha_k * A_k)
    last_conv_output = last_conv_layer_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Apply ReLU to retain only features that positively influence the target class
    heatmap = tf.maximum(heatmap, 0.0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(raw_img_pil, heatmap, alpha=0.4):
    img = np.array(raw_img_pil)
    
    # Rescale heatmap to [0, 255]
    heatmap_255 = np.uint8(255 * heatmap)
    
    # Apply JET colormap
    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_255]
    
    # Resize heatmap to match image dimensions
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = tf.keras.preprocessing.image.img_to_array(jet_heatmap)
    
    # Blend overlay
    superimposed_img = jet_heatmap * alpha + img
    superimposed_img = np.uint8(tf.clip_by_value(superimposed_img, 0, 255))
    return superimposed_img


## 5. Grad-CAM Diagnostics on Test Foliage Specimens
We test Grad-CAM across multiple distinct plant pathology classes to verify that the model attends to genuine disease spots.


In [ ]:
test_sample_classes = [
    'Tomato___Early_blight',
    'Tomato___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Late_blight'
]

plt.figure(figsize=(16, 12))
plot_idx = 1

for cls_name in test_sample_classes:
    cls_dir = os.path.join("data", "plant_village_5_classes", "test", cls_name)
    sample_img_name = os.listdir(cls_dir)[0]
    p = os.path.join(cls_dir, sample_img_name)
    
    pil_img = Image.open(p).convert('RGB').resize((224, 224))
    img_arr = preprocess_input(np.expand_dims(np.array(pil_img, dtype=np.float32), 0))
    
    preds = model.predict(img_arr, verbose=0)[0]
    top_pred_idx = np.argmax(preds)
    pred_label = classes[top_pred_idx]
    confidence = preds[top_pred_idx]
    
    # Generate Grad-CAM heatmap
    heatmap = make_gradcam_heatmap(img_arr, model, last_conv_layer_name="conv5_block3_out")
    overlay = overlay_gradcam(pil_img, heatmap, alpha=0.45)
    
    # Plot original
    plt.subplot(len(test_sample_classes), 3, plot_idx)
    plt.imshow(pil_img)
    plt.title(f"True: {cls_name}\nPred: {pred_label} ({confidence*100:.1f}%)", fontsize=9)
    plt.axis('off')
    
    # Plot Heatmap
    plt.subplot(len(test_sample_classes), 3, plot_idx + 1)
    plt.imshow(heatmap, cmap='jet')
    plt.title("Grad-CAM Heatmap", fontsize=9)
    plt.axis('off')
    
    # Plot Overlay
    plt.subplot(len(test_sample_classes), 3, plot_idx + 2)
    plt.imshow(overlay)
    plt.title("Superimposed Salience", fontsize=9)
    plt.axis('off')
    
    plot_idx += 3

plt.suptitle("Grad-CAM Salience Visualizations on Plant Foliage", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 6. Interpretability Discussion & Limitations
* **Salience Alignment**: On diseased leaves, the Grad-CAM heatmap focuses sharply on chlorotic rings, necrosis margins, and fungal spots. On healthy specimens, attention is diffusely distributed across green lamina tissue.
* **Limitations**: Grad-CAM visualizes which spatial regions feature high activation times gradient; it does **not** provide mathematical proof of causal necessity. Background artifacts (e.g. soil or bench markers) can occasionally influence activations, highlighting why field validation is critical.
